## ML Classical Baselines

In [ ]:
import pandas as pd
from IPython.display import display

from utils.config import (
    BANDS_TO_RUN,
    DATA_DIR,
    DEFAULT_OVERLAP_SIZE,
    DEFAULT_WINDOW_SIZE,
)
from utils.ML.ml_pipeline import (
    best_confusion_predictions,
    get_cache_path,
    get_results_path,
    load_all_predictions,
    load_feature_dataframes,
    load_lovo_summary_tables,
    load_or_build_none_reference_features,
    load_params_lookup,
    load_raw_csi_data,
    lovo_aggregated_analysis_table,
    master_results_table,
    per_room_position_accuracy_table,
    print_normalization_discriminability,
    run_global_baselines,
    run_optional_grid_search,
    save_analysis_tables,
    save_lovo_analysis_table,
)
from utils.plots import (
    plot_band_error_cdf,
    plot_block_vs_lovo_floor_plan,
    plot_block_vs_lovo_metrics,
    plot_csi_magnitude_stages,
    plot_floor_plan_heatmap,
    plot_global_position_confusion_matrix,
    plot_localization_error_cdf_by_model,
    plot_lovo_cdf_triptych,
    plot_lovo_volunteer_variability,
    plot_model_band_error_boxplot,
    plot_position_confusion_by_true_room,
)

#### Project Configurations

In [ ]:
CALIBRATION_MODE = "rssi"   # ("none", "packet_norm", "rssi")

CSV_PROCESSING_OPTIONS = {
    "max_workers": 1,
    "cache_dir": None,
    "use_cache": True,
    "force_reprocess": False,
    "min_rssi_dbm": -95.0,
    "calibration_eps": 1e-12,
}

MAGNITUDE_PROCESSING_OPTIONS = {
    "normalization": "empty_baseline"
    ,  # none | zscore | minmax | packet_minmax | empty_baseline
    "epsilon": 1e-8,
}
NORMALIZATION_BASELINE_SCOPE = "per_session"  # per_session | per_user | global (empty_baseline only)
SHOW_MAGNITUDE_PLOT = False

MAGNITUDE_PLOT_OPTIONS = {
    "position": "D-8",
    "user": "03",
    "trial": "01",
    "anchor_pair": 7,
    "packet_count": 60,
    "packet_selection": "middle",
    "normalized_limit_percentile": 99.0,
    "surface_elevation": 28,
    "surface_azimuth": -135,
    "save": False,
    "output_directory": "outputs/magnitude_plots",
}

FEATURE_EXTRACTION_OPTIONS = {
    "window_size": DEFAULT_WINDOW_SIZE,
    "overlap_size": DEFAULT_OVERLAP_SIZE,
    "require_all_esps": False,
}

MODELS_TO_RUN = ("RF", "KNN", "SVM")
SPLIT_MODES = ("block", "lovo")  # cross_session, lovo, random, block
RUN_CLASSIFICATION = False
RUN_GRID_SEARCH = False
REQUIRE_TUNED_PARAMS = True  # True -> require an exact matching grid-search run
FORCE_RETRAIN = False
SAVE_PREDICTIONS = True
N_JOBS = 8

BLOCK_COUNT = 10
TEST_SIZE = 0.30
RANDOM_STATE = 42
ROW_SPACING = 1.0
COLUMN_SPACING = 1.0
SVM_FUSION_FALLBACK_SECONDS = 30 * 60

CONFUSION_DATASET = "Fusion"
CONFUSION_MODEL = "best"      # "best", "RF", "KNN", or "SVM"

SHOW_CDF_BY_BAND = True
SHOW_CDF_BY_MODEL = True
SHOW_BOXPLOT = True
SHOW_FLOOR_PLAN = True
SHOW_CONFUSION_MATRICES = True
SHOW_PER_ROOM_PLOTS = False

preproc_opts = dict(MAGNITUDE_PROCESSING_OPTIONS)
if preproc_opts.get("normalization") == "empty_baseline":
    preproc_opts["baseline_scope"] = NORMALIZATION_BASELINE_SCOPE
feat_opts = dict(FEATURE_EXTRACTION_OPTIONS)
feature_cache_dir = get_cache_path(preproc_opts, feat_opts)
results_dir = get_results_path()
print(f"Feature cache path: {feature_cache_dir}")
print(f"Results path: {results_dir}")
tables_dir = results_dir / "tables"
plots_dir = results_dir / "plots"
for directory in (tables_dir, plots_dir):
    directory.mkdir(parents=True, exist_ok=True)


def _slugify(value: str) -> str:
    """Convert a display value to a compact filename-safe slug."""
    return value.lower().replace(".", "-").replace(" ", "-").strip("-")

## Raw Data

In [ ]:
loaded_csi_data = load_raw_csi_data(
    DATA_DIR,
    calibration_mode=CALIBRATION_MODE,
    csv_options=CSV_PROCESSING_OPTIONS,
    return_magnitude_stages=SHOW_MAGNITUDE_PLOT,
)
magnitude_data, csv_diagnostics = loaded_csi_data[:2]
magnitude_plot_stages = loaded_csi_data[2] if SHOW_MAGNITUDE_PLOT else None

In [ ]:
if SHOW_MAGNITUDE_PLOT:
    plot_csi_magnitude_stages(magnitude_plot_stages, **MAGNITUDE_PLOT_OPTIONS)

#### Feature Dataframes

In [ ]:
feature_dataframes = load_feature_dataframes(
    magnitude_data,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
    bands_to_run=BANDS_TO_RUN,
)

none_reference_dataframes = (
    feature_dataframes
    if preproc_opts.get("normalization") == "none"
    else load_or_build_none_reference_features(
        magnitude_data,
        active_preproc_opts=preproc_opts,
        feat_opts=feat_opts,
    )
)
fisher_diagnostics = print_normalization_discriminability(
    feature_dataframes,
    normalization=preproc_opts.get("normalization", "none"),
    reference_feature_dataframes=none_reference_dataframes,
    bands_to_run=BANDS_TO_RUN,
)
display(fisher_diagnostics)
del magnitude_data


#### Model Parameters

In [ ]:
params_lookup = {}
if RUN_GRID_SEARCH:
    print("Parameter lookup deferred until the requested grid search completes.")
else:
    params_lookup = load_params_lookup(
        results_dir,
        models_to_run=MODELS_TO_RUN,
        bands_to_run=BANDS_TO_RUN,
        preproc_opts=preproc_opts,
        feat_opts=feat_opts,
        require_tuned_params=REQUIRE_TUNED_PARAMS,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        n_blocks=BLOCK_COUNT,
    )
    for key, value in params_lookup.items():
        print(f"{key}: {value}")

##### Optional Grid Search

In [ ]:
grid_ran = run_optional_grid_search(
    feature_dataframes,
    run_grid_search=RUN_GRID_SEARCH,
    results_dir=results_dir,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
    models_to_run=MODELS_TO_RUN,
    bands_to_run=BANDS_TO_RUN,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    n_blocks=BLOCK_COUNT,
    row_spacing=ROW_SPACING,
    column_spacing=COLUMN_SPACING,
)
if grid_ran:
    raise SystemExit("RUN_GRID_SEARCH=True completed; set it to False before running experiments.")

#### Global 52-Class Baselines

In [ ]:
if RUN_CLASSIFICATION:
    global_summary, global_predictions_by_key = run_global_baselines(
        feature_dataframes,
        params_lookup=params_lookup,
        models_to_run=MODELS_TO_RUN,
        bands_to_run=BANDS_TO_RUN,
        split_modes=SPLIT_MODES,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        n_blocks=BLOCK_COUNT,
        n_jobs=N_JOBS,
        results_dir=results_dir,
        preproc_opts=preproc_opts,
        feat_opts=feat_opts,
        force_retrain=FORCE_RETRAIN,
        save_predictions=SAVE_PREDICTIONS,
        svm_fallback_seconds=SVM_FUSION_FALLBACK_SECONDS,
        row_spacing=ROW_SPACING,
        column_spacing=COLUMN_SPACING,
    )
    display(global_summary)
    run_registry = pd.read_csv(results_dir / "runs.csv")
    active_run_ids = run_registry.loc[
        run_registry["model"].isin([model.lower() for model in MODELS_TO_RUN])
        & run_registry["split"].isin(SPLIT_MODES),
        "run_id",
    ]
    if active_run_ids.empty:
        raise RuntimeError("No active run_id is available for plot storage.")
    plots_dir = results_dir / "plots" / active_run_ids.iloc[0]
    plots_dir.mkdir(parents=True, exist_ok=True)
else:
    print("Classification skipped because RUN_CLASSIFICATION is False.")


#### Analysis Tables

Tables are grouped by evaluation split: Temporal Block, Leave-One-Volunteer-Out (LOVO), and Block versus LOVO (shown only when both splits are available).

In [ ]:
# Set to False to save every figure below without rendering it inline (e.g. non-interactive/batch runs).
DISPLAY_PLOTS = True

all_global_predictions = load_all_predictions(
    results_dir,
    models_to_run=MODELS_TO_RUN,
    bands_to_run=BANDS_TO_RUN,
    split_modes=SPLIT_MODES,
)

master_table = master_results_table(
    all_global_predictions,
    summary_path=results_dir / "runs.csv",
)
per_room_table = per_room_position_accuracy_table(all_global_predictions)
save_analysis_tables(master_table, per_room_table, tables_dir=tables_dir)

available_splits = set(master_table["split"].astype(str)) if not master_table.empty else set()
has_block = "block" in SPLIT_MODES and "block" in available_splits

# load_lovo_summary_tables() reads every LOVO row in runs.csv without a model filter, so
# it can also return DL (CNN) runs saved by the other notebook; keep this classical-ML-only.
ml_model_set = {model.upper() for model in MODELS_TO_RUN}
lovo_per_fold = pd.DataFrame()
lovo_summary = pd.DataFrame()
lovo_table = pd.DataFrame()
has_lovo = False
if "lovo" in SPLIT_MODES:
    try:
        raw_lovo_per_fold, raw_lovo_summary = load_lovo_summary_tables(results_dir)
        lovo_summary = raw_lovo_summary.loc[
            raw_lovo_summary["model"].astype(str).str.upper().isin(ml_model_set)
        ].reset_index(drop=True)
        lovo_per_fold = raw_lovo_per_fold.loc[
            raw_lovo_per_fold["model"].astype(str).str.upper().isin(ml_model_set)
        ].reset_index(drop=True)
        if lovo_summary.empty:
            raise ValueError("No classical-ML LOVO runs remain after excluding DL results.")
        lovo_table = lovo_aggregated_analysis_table(lovo_summary)
        save_lovo_analysis_table(lovo_table, tables_dir=tables_dir)
        has_lovo = True
    except (FileNotFoundError, ValueError) as error:
        print(f"LOVO analysis skipped: {error}")

print(f"Temporal Block available: {has_block} | LOVO available: {has_lovo}")

##### Temporal Block

In [ ]:
if has_block:
    display(master_table.loc[master_table["split"] == "block"].reset_index(drop=True))
    display(per_room_table.loc[per_room_table["split"] == "block"].reset_index(drop=True))
else:
    print("Temporal Block tables skipped: the block split is not selected or unavailable.")

##### Leave-One-Volunteer-Out (LOVO)

In [ ]:
if has_lovo:
    print("Fold mean +/- fold standard deviation, per model x band:")
    display(lovo_table)

    print("Per-fold position accuracy (RF):")
    display(
        lovo_per_fold.loc[lovo_per_fold["model"].str.upper() == "RF"].reset_index(drop=True)
    )

    pooled_columns = [
        column
        for column in ("model", "dataset", "position_accuracy_pooled")
        if column in lovo_summary.columns
    ]
    if pooled_columns:
        print("Pooled position accuracy (all held-out predictions combined, not fold-averaged):")
        display(
            lovo_summary[pooled_columns].sort_values(["dataset", "model"]).reset_index(drop=True)
        )

    majority_columns = [
        column
        for column in (
            "model",
            "dataset",
            "majority_position_accuracy_mean_std",
            "majority_room_accuracy_mean_std",
        )
        if column in lovo_table.columns
    ]
    if majority_columns:
        print("Majority-class baseline (fold mean +/- fold standard deviation):")
        display(lovo_table[majority_columns])
else:
    print("LOVO tables skipped: the LOVO split is not selected or unavailable.")

##### Block versus LOVO

In [ ]:
if has_block and has_lovo:
    comparison_rows = []
    for _, block_row in master_table.loc[master_table["split"] == "block"].iterrows():
        lovo_row = lovo_summary.loc[
            (lovo_summary["model"] == block_row["model"])
            & (lovo_summary["dataset"] == block_row["dataset"])
        ]
        if lovo_row.empty:
            continue
        lovo_row = lovo_row.iloc[0]
        comparison_rows.append(
            {
                "model": block_row["model"],
                "dataset": block_row["dataset"],
                "block_position_accuracy": block_row["position_accuracy"],
                "lovo_position_accuracy_mean": lovo_row["position_accuracy_mean"],
                "lovo_position_accuracy_std": lovo_row["position_accuracy_std"],
                "generalization_gap": (
                    block_row["position_accuracy"] - lovo_row["position_accuracy_mean"]
                ),
            }
        )
    block_vs_lovo_table = (
        pd.DataFrame(comparison_rows).sort_values(["dataset", "model"]).reset_index(drop=True)
    )
    display(block_vs_lovo_table)
else:
    print("Block vs LOVO comparison table skipped: both splits are required.")

#### Temporal Block Plots

In [ ]:
if has_block:
    block_predictions = all_global_predictions.loc[all_global_predictions["split_mode"] == "block"]

    if SHOW_CDF_BY_BAND:
        for band in BANDS_TO_RUN:
            plot_localization_error_cdf_by_model(
                block_predictions,
                dataset=band,
                save_path=plots_dir / f"cdf_by_model_block_{_slugify(band)}.png",
                show=DISPLAY_PLOTS,
            )

    if SHOW_CDF_BY_MODEL:
        for model in MODELS_TO_RUN:
            model_predictions = block_predictions.loc[block_predictions["model"] == model]
            plot_band_error_cdf(
                model_predictions,
                model_label=model,
                split_modes=("block",),
                band_order=BANDS_TO_RUN,
                save_path=plots_dir,
                show=DISPLAY_PLOTS,
            )

    if SHOW_BOXPLOT:
        plot_model_band_error_boxplot(
            block_predictions,
            models=MODELS_TO_RUN,
            bands=BANDS_TO_RUN,
            save_path=plots_dir / "boxplot_block_model_band_distance_error.png",
            show=DISPLAY_PLOTS,
        )

    block_confusion_model, block_confusion_predictions = best_confusion_predictions(
        all_global_predictions,
        master_table.loc[master_table["split"] == "block"],
        dataset=CONFUSION_DATASET,
        model=CONFUSION_MODEL,
        split="block",
    )
    print(f"Temporal Block confusion/floor-plan model: {block_confusion_model} on {CONFUSION_DATASET}")

    if SHOW_FLOOR_PLAN:
        plot_floor_plan_heatmap(
            block_confusion_predictions,
            title=f"Temporal Block - {CONFUSION_DATASET} / {block_confusion_model} spatial performance",
            save_path=(
                plots_dir
                / f"floor_plan_block_{_slugify(CONFUSION_DATASET)}_{_slugify(block_confusion_model)}.png"
            ),
            show=DISPLAY_PLOTS,
        )

    if SHOW_CONFUSION_MATRICES:
        plot_global_position_confusion_matrix(
            block_confusion_predictions,
            dataset=CONFUSION_DATASET,
            normalize="true",
            save_path=(
                plots_dir
                / f"confusion_block_{_slugify(CONFUSION_DATASET)}_{_slugify(block_confusion_model)}.png"
            ),
            show=DISPLAY_PLOTS,
        )
        if SHOW_PER_ROOM_PLOTS:
            room_plot_dir = (
                plots_dir
                / f"confusion_by_room_block_{_slugify(CONFUSION_DATASET)}_{_slugify(block_confusion_model)}"
            )
            plot_position_confusion_by_true_room(
                block_confusion_predictions,
                dataset=CONFUSION_DATASET,
                normalize="true",
                save_path=room_plot_dir,
                show=DISPLAY_PLOTS,
            )
else:
    print("Temporal Block plots skipped: the block split is not selected or unavailable.")

#### LOVO Plots

In [ ]:
if has_lovo:
    lovo_predictions = all_global_predictions.loc[all_global_predictions["split_mode"] == "lovo"]
    lovo_model_slug = "-".join(_slugify(model) for model in MODELS_TO_RUN)

    plot_lovo_cdf_triptych(
        lovo_predictions,
        models=MODELS_TO_RUN,
        band_order=BANDS_TO_RUN,
        save_path=plots_dir / f"cdf_triptych_lovo_{lovo_model_slug}.png",
        show=DISPLAY_PLOTS,
    )

    plot_lovo_volunteer_variability(
        lovo_per_fold,
        bands=BANDS_TO_RUN,
        model="RF",
        save_path=plots_dir / f"lovo_{_slugify('RF')}_variability_slope.png",
        show=DISPLAY_PLOTS,
    )

    lovo_fusion_rf_predictions = lovo_predictions.loc[
        (lovo_predictions["model"] == "RF") & (lovo_predictions["dataset"] == "Fusion")
    ]
    plot_floor_plan_heatmap(
        lovo_fusion_rf_predictions,
        title="LOVO - Fusion / RF spatial performance",
        save_path=plots_dir / f"floor_plan_lovo_{_slugify('Fusion')}_{_slugify('RF')}.png",
        show=DISPLAY_PLOTS,
    )
else:
    print("LOVO plots skipped: the LOVO split is not selected or unavailable.")

#### Block versus LOVO Plots

In [ ]:
if has_block and has_lovo:
    plot_block_vs_lovo_metrics(
        master_table.loc[master_table["split"] == "block"],
        lovo_summary,
        bands=BANDS_TO_RUN,
        model="RF",
        save_path=plots_dir / f"block_vs_lovo_{_slugify('RF')}_metrics.png",
        show=DISPLAY_PLOTS,
    )

    block_predictions = all_global_predictions.loc[all_global_predictions["split_mode"] == "block"]
    lovo_predictions = all_global_predictions.loc[all_global_predictions["split_mode"] == "lovo"]
    block_fusion_rf_predictions = block_predictions.loc[
        (block_predictions["model"] == "RF") & (block_predictions["dataset"] == "Fusion")
    ]
    lovo_fusion_rf_predictions = lovo_predictions.loc[
        (lovo_predictions["model"] == "RF") & (lovo_predictions["dataset"] == "Fusion")
    ]
    plot_block_vs_lovo_floor_plan(
        block_fusion_rf_predictions,
        lovo_fusion_rf_predictions,
        title="Temporal Block vs LOVO - Fusion / RF spatial position accuracy",
        save_path=plots_dir / f"floor_plan_block_vs_lovo_{_slugify('Fusion')}_{_slugify('RF')}.png",
        show=DISPLAY_PLOTS,
    )
else:
    print("Temporal Block vs LOVO plots skipped: both splits are required.")